# Ensemble Learning & Maintenance Intelligence

This notebook focuses on ensemble learning for CampusPulse complaint priority prediction and the development of an intelligent maintenance ranking system.

In [29]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "campuspulse_akgec_cleaned.csv"
PREPROCESSOR_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "campuspulse_preprocessor.pkl"
PCA_PATH = PROJECT_ROOT / "ml" / "models" / "classification" / "campuspulse_pca.pkl"
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)
print("Priority:")
print(df["priority"].value_counts())

Dataset: (200000, 37)
Priority:
priority
Medium      70197
Low         68808
High        42242
Critical    18753
Name: count, dtype: int64


## 2. Dataset and Preprocessing

The CampusPulse dataset contains complaint records with information related to complaint characteristics, infrastructure, recurrence, affected users and priority.

The preprocessing pipeline developed for the classification task is reused in this notebook to maintain consistency with the existing ML workflow.

## 3. Feature Engineering

Additional features are derived from complaint date, time, recent complaint frequency, recurrence, affected users, infrastructure age and operational pressure.

These features help capture temporal patterns, complaint growth and potential impact.

In [5]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"], errors="coerce")
df["complaint_month"] = df["complaint_date"].dt.month
df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
df["complaint_year"] = df["complaint_date"].dt.year
df["is_weekend"] = (df["complaint_dayofweek"] >= 5).astype(int)

df["complaint_time"] = pd.to_datetime(df["complaint_time"], errors="coerce")
df["complaint_hour"] = df["complaint_time"].dt.hour

df["time_period"] = pd.cut(
    df["complaint_hour"],
    bins=[-1, 5, 11, 17, 21, 24],
    labels=["Night", "Morning", "Afternoon", "Evening", "Late_Night"]
)

df["total_recent_complaints"] = (
    df["complaints_last_7_days"].fillna(0) +
    df["complaints_last_30_days"].fillna(0)
)

df["complaint_growth"] = (
    df["complaints_last_7_days"].fillna(0) /
    (df["complaints_last_30_days"].fillna(0) + 1)
)

df["impact_score"] = (
    df["affected_users"].fillna(0) *
    (1 + df["recurrence_count"].fillna(0))
)

df["duration_per_user"] = (
    df["issue_duration_hours"].fillna(0) /
    (df["affected_users"].fillna(0) + 1)
)

df["repair_cost_per_user"] = (
    df["estimated_repair_cost"].fillna(0) /
    (df["affected_users"].fillna(0) + 1)
)

df["equipment_infrastructure_age_gap"] = (
    df["infrastructure_age_years"].fillna(0) -
    df["equipment_age_years"].fillna(0)
)

df["operational_pressure"] = (
    df["complaints_last_7_days"].fillna(0) +
    df["recurrence_count"].fillna(0) +
    df["affected_users"].fillna(0) / 10
)

print("Feature engineering done:", df.shape)

C:\Users\v2126\AppData\Local\Temp\ipykernel_4888\1718437082.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(df["complaint_time"], errors="coerce")


Feature engineering done: (200000, 50)


In [6]:
df.drop(columns=["complaint_date", "complaint_time"], inplace=True)

leakage_cols = [
    "complaint_status",
    "response_time_hours",
    "resolution_time_hours",
    "maintenance_team",
    "resolution_type",
    "actual_expenditure"
]

df.drop(columns=leakage_cols, inplace=True)
df.drop(columns=["complaint_id"], inplace=True)

X = df.drop(columns="priority")
y = df["priority"]

print("X:", X.shape)
print("y:", y.shape)

X: (200000, 40)
y: (200000,)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (160000, 40)
Test: (40000, 40)


In [8]:
preprocessor = joblib.load(PREPROCESSOR_PATH)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed test:", X_test_processed.shape)

Processed train: (160000, 304)
Processed test: (40000, 304)


In [9]:
pca = joblib.load(PCA_PATH)

X_train_pca = pca.transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)

print("PCA train:", X_train_pca.shape)
print("PCA test:", X_test_pca.shape)

PCA train: (160000, 152)
PCA test: (40000, 152)


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

print("BASE MODELS OK")

BASE MODELS OK


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Classes:", le.classes_)

Classes: ['Critical' 'High' 'Low' 'Medium']


In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Classes:", le.classes_)

Classes: ['Critical' 'High' 'Low' 'Medium']


In [13]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_pca, y_train_enc)

lr_pred = lr_model.predict(X_test_pca)

print("Logistic Regression done")

Logistic Regression done


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test_enc, lr_pred),
    "Precision": precision_score(y_test_enc, lr_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, lr_pred, average="weighted"),
    "F1": f1_score(y_test_enc, lr_pred, average="weighted")
}

print(lr_results)

{'Model': 'Logistic Regression', 'Accuracy': 0.6484, 'Precision': 0.6708972255716052, 'Recall': 0.6484, 'F1': 0.6473293475891866}


In [15]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=15
)

dt_model.fit(X_train_pca, y_train_enc)

dt_pred = dt_model.predict(X_test_pca)

print("Decision Tree done")

Decision Tree done


In [16]:
dt_results = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test_enc, dt_pred),
    "Precision": precision_score(y_test_enc, dt_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, dt_pred, average="weighted"),
    "F1": f1_score(y_test_enc, dt_pred, average="weighted")
}

print(dt_results)

{'Model': 'Decision Tree', 'Accuracy': 0.58375, 'Precision': 0.5898846802356943, 'Recall': 0.58375, 'F1': 0.5814330705007481}


In [17]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

xgb_model.fit(X_train_pca, y_train_enc)

xgb_pred = xgb_model.predict(X_test_pca)

print("XGBoost done")

XGBoost done


In [18]:
xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test_enc, xgb_pred),
    "Precision": precision_score(y_test_enc, xgb_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, xgb_pred, average="weighted"),
    "F1": f1_score(y_test_enc, xgb_pred, average="weighted")
}

print(xgb_results)

{'Model': 'XGBoost', 'Accuracy': 0.637375, 'Precision': 0.6533444336224776, 'Recall': 0.637375, 'F1': 0.6363904824017421}


In [19]:
import numpy as np

hard_vote_pred = np.apply_along_axis(
    lambda x: np.bincount(x, minlength=4).argmax(),
    axis=0,
    arr=np.vstack([lr_pred, dt_pred, xgb_pred])
)

print("Hard Voting done")

Hard Voting done


In [20]:
hard_voting_results = {
    "Model": "Hard Voting",
    "Accuracy": accuracy_score(y_test_enc, hard_vote_pred),
    "Precision": precision_score(y_test_enc, hard_vote_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, hard_vote_pred, average="weighted"),
    "F1": f1_score(y_test_enc, hard_vote_pred, average="weighted")
}

print(hard_voting_results)

{'Model': 'Hard Voting', 'Accuracy': 0.64245, 'Precision': 0.6607986003626002, 'Recall': 0.64245, 'F1': 0.6415749466474406}


In [21]:
lr_proba = lr_model.predict_proba(X_test_pca)
dt_proba = dt_model.predict_proba(X_test_pca)
xgb_proba = xgb_model.predict_proba(X_test_pca)

soft_vote_proba = (
    lr_proba +
    dt_proba +
    xgb_proba
) / 3

soft_vote_pred = np.argmax(soft_vote_proba, axis=1)

print("Soft Voting done")

Soft Voting done


In [22]:
soft_voting_results = {
    "Model": "Soft Voting",
    "Accuracy": accuracy_score(y_test_enc, soft_vote_pred),
    "Precision": precision_score(y_test_enc, soft_vote_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, soft_vote_pred, average="weighted"),
    "F1": f1_score(y_test_enc, soft_vote_pred, average="weighted")
}

print(soft_voting_results)

{'Model': 'Soft Voting', 'Accuracy': 0.61915, 'Precision': 0.6281140195449415, 'Recall': 0.61915, 'F1': 0.617707160728443}


In [23]:
bagging_preds = []

for i in range(5):
    rng = np.random.RandomState(42 + i)
    idx = rng.choice(len(X_train_pca), size=50000, replace=True)

    model = DecisionTreeClassifier(
        max_depth=12,
        random_state=42 + i
    )

    model.fit(X_train_pca[idx], y_train_enc[idx])
    bagging_preds.append(model.predict(X_test_pca))

bagging_pred = np.apply_along_axis(
    lambda x: np.bincount(x, minlength=4).argmax(),
    axis=0,
    arr=np.array(bagging_preds)
)

print("Bagging done")

Bagging done


In [24]:
bagging_results = {
    "Model": "Bagging",
    "Accuracy": accuracy_score(y_test_enc, bagging_pred),
    "Precision": precision_score(y_test_enc, bagging_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, bagging_pred, average="weighted"),
    "F1": f1_score(y_test_enc, bagging_pred, average="weighted")
}

print(bagging_results)

{'Model': 'Bagging', 'Accuracy': 0.59555, 'Precision': 0.6051913669789696, 'Recall': 0.59555, 'F1': 0.5938451453677052}


In [25]:
boosting_pred = xgb_pred

boosting_results = {
    "Model": "Boosting (XGBoost)",
    "Accuracy": accuracy_score(y_test_enc, boosting_pred),
    "Precision": precision_score(y_test_enc, boosting_pred, average="weighted"),
    "Recall": recall_score(y_test_enc, boosting_pred, average="weighted"),
    "F1": f1_score(y_test_enc, boosting_pred, average="weighted")
}

print(boosting_results)

{'Model': 'Boosting (XGBoost)', 'Accuracy': 0.637375, 'Precision': 0.6533444336224776, 'Recall': 0.637375, 'F1': 0.6363904824017421}


In [26]:
all_results = pd.DataFrame([
    lr_results,
    dt_results,
    xgb_results,
    hard_voting_results,
    soft_voting_results,
    bagging_results,
    boosting_results
])

all_results.sort_values("F1", ascending=False).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.648400,0.670897,0.648400,0.647329
1,Hard Voting,0.642450,0.660799,0.642450,0.641575
2,XGBoost,0.637375,0.653344,0.637375,0.636390
3,Boosting (XGBoost),0.637375,0.653344,0.637375,0.636390
4,Soft Voting,0.619150,0.628114,0.619150,0.617707
5,Bagging,0.595550,0.605191,0.595550,0.593845
6,Decision Tree,0.583750,0.589885,0.583750,0.581433


## 9. Conclusion

The experimental results show that Logistic Regression achieved the best overall performance among the evaluated models, with an Accuracy of 64.84% and a weighted F1-score of 0.6473.

Hard Voting achieved an Accuracy of 64.25% and a weighted F1-score of 0.6416, while XGBoost achieved 63.74% Accuracy and a weighted F1-score of 0.6364.

Soft Voting and Bagging produced lower performance compared with the individual Logistic Regression model. Therefore, for the current dataset and feature representation, Logistic Regression provides the strongest classification performance among the evaluated approaches.

The Maintenance Intelligence layer will further combine predicted priority with complaint recurrence, similar complaint frequency and affected-user count to improve complaint prioritization for maintenance planning.

In [27]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

print("XGBoost import OK")

XGBoost import OK


In [28]:
from sklearn.model_selection import RandomizedSearchCV

xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

param_grid = {
    "n_estimators": [150, 250, 350],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,
    scoring="f1_weighted",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=1
)

xgb_search.fit(X_train_pca, y_train_enc)

print("Best Parameters:")
print(xgb_search.best_params_)
print("Best CV F1:", xgb_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


Best Parameters:
{'subsample': 0.7, 'n_estimators': 350, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.8}
Best CV F1: 0.6348896483074
